# Analítica de Datos — Tema 1: Preprocesamiento de datos (Nivel Licenciatura)
## Notebook teórico-práctica (NumPy + Pandas) — Versión básica

**Propósito:** aprender los fundamentos del preprocesamiento de datos con ejemplos sencillos y prácticos.

**Perfil esperado del estudiante:** ya conoce lo básico de `NumPy` y `Pandas`.

## ¿Qué aprenderás hoy?

- Identificar tipos de datos en un `DataFrame`
- Revisar un archivo con datos crudos (`CSV`)
- Detectar problemas comunes (faltantes, duplicados, errores de escritura, valores fuera de rango)
- Aplicar limpieza básica con Pandas
- Filtrar, transformar y unir tablas con `merge()`
- Guardar un archivo limpio

## Recomendación de uso en clase
1. Ejecuta una celda a la vez.  
2. Lee primero el comentario.  
3. Responde las preguntas cortas.  
4. Resuelve el mini reto al final.

In [ ]:
# 1) Preparación del entorno
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print("Entorno listo ✅")
print("Pandas:", pd.__version__)
print("NumPy :", np.__version__)

## 1.1 ¿Qué es el preprocesamiento de datos? (muy breve)

Es el proceso de **revisar, limpiar y transformar datos** antes de analizarlos.

### Problemas comunes
- texto inconsistente (`SLP Centro` vs `slp centro`)
- valores faltantes
- duplicados
- porcentajes > 100
- errores lógicos (por ejemplo, créditos aprobados > inscritos)

> Primero limpiamos; después analizamos.

## 2) Creamos un dataset de práctica con errores

Esto nos permite practicar un caso realista sin depender de archivos externos.

In [ ]:
# 2) Generar datos de práctica (con errores intencionales)
Path("datos_basicos").mkdir(exist_ok=True)

df_crudo = pd.DataFrame({
    "id_estudiante": [101,102,103,104,105,106,107,108,109,110,110],
    "nombre": ["Ana","Luis","Marta","José","Elena","Carlos","Rosa","Miguel","Diana","Pablo","Pablo"],
    "edad": [19, 20, 21, np.nan, 22, 19, 20, 45, 21, 20, 20],
    "campus": ["SLP Centro", "slp centro ", "SLP Norte", "SLP Cntro", "Matehuala", "SLP Norte", "Matehuala", "SLP Centro", "SLP Norte", "SLP Centro", "SLP Centro"],
    "modalidad": ["Escolarizada", "escolar", "Mixta", "MIXTA ", "Escolarizada", "Escolarizada", "Mixta", "Mixta", "Escolarizada", "Escolarizada", "Escolarizada"],
    "promedio": [88, 91, np.nan, 76, 84, 130, 69, 72, 95, 81, 81],
    "asistencia_pct": [92, 85, 78, 88, 101, 90, 73, 80, np.nan, 95, 95],
    "creditos_inscritos": [30, 32, 28, 30, 26, 30, 24, 30, 32, 28, 28],
    "creditos_aprobados": [28, 30, 26, 32, 25, 27, 20, 29, 31, 28, 28],
})

df_apoyos = pd.DataFrame({
    "id_estudiante": [101,103,104,106,108,109],
    "beca": ["Sí","No","Sí","No","Sí","No"],
    "tutoria_horas_mes": [4, 0, 2, 1, 3, 0]
})

df_crudo.to_csv("datos_basicos/estudiantes_crudo.csv", index=False)
df_apoyos.to_csv("datos_basicos/apoyos.csv", index=False)

print("Archivos creados:")
print("- datos_basicos/estudiantes_crudo.csv")
print("- datos_basicos/apoyos.csv")
display(df_crudo)

## 3) Importación e inspección inicial
Antes de limpiar, observamos el conjunto de datos.

In [ ]:
df = pd.read_csv("datos_basicos/estudiantes_crudo.csv")
apoyos = pd.read_csv("datos_basicos/apoyos.csv")

print("Tamaño de df:", df.shape)
print("Tamaño de apoyos:", apoyos.shape)
display(df.head())

In [ ]:
df.info()

In [ ]:
df.describe()

### Preguntas rápidas
1. ¿Qué columnas son numéricas?
2. ¿Qué columnas son de texto?
3. ¿Cuál es el identificador?

## 4) Tipos de datos
- Numéricos: `edad`, `promedio`, `asistencia_pct`
- Texto/categorías: `campus`, `modalidad`
- Identificador: `id_estudiante`

In [ ]:
df.dtypes

## 5) Cuantificación básica (conteos y proporciones)

In [ ]:
# Valores faltantes por columna
df.isna().sum()

In [ ]:
# Frecuencia de campus
df["campus"].value_counts(dropna=False)

In [ ]:
# Frecuencia de modalidad
df["modalidad"].value_counts(dropna=False)

In [ ]:
# Proporciones de modalidad
df["modalidad"].value_counts(normalize=True, dropna=False)

## 6) Muestreo (`sample`)
Sirve para revisar rápidamente una parte de los datos.

In [ ]:
df.sample(4, random_state=10)

## 7) Similitud (introducción sencilla)
Usaremos distancia euclidiana para comparar dos registros numéricos.

In [ ]:
cols = ["edad", "promedio", "asistencia_pct"]
tmp = df[cols].dropna().head(2).copy()

a = tmp.iloc[0].to_numpy(dtype=float)
b = tmp.iloc[1].to_numpy(dtype=float)

distancia = np.linalg.norm(a - b)

print("Fila A:", a)
print("Fila B:", b)
print("Distancia euclidiana:", round(float(distancia), 3))
display(tmp)

> Menor distancia = mayor parecido (en esas variables).

## 8) Detección de errores en los datos
Buscaremos:
1. faltantes  
2. duplicados  
3. texto inconsistente  
4. valores fuera de rango  
5. errores lógicos

In [ ]:
# 8.1 Faltantes
print("Valores faltantes por columna:")
display(df.isna().sum())

print("Filas con al menos un faltante:")
display(df[df.isna().any(axis=1)])

In [ ]:
# 8.2 Duplicados por id_estudiante
duplicados_id = df[df.duplicated(subset=["id_estudiante"], keep=False)]

print("Registros duplicados por id_estudiante:", len(duplicados_id))
display(duplicados_id.sort_values("id_estudiante"))

In [ ]:
# 8.3 Texto inconsistente
print("Valores únicos de campus:")
print(sorted(df["campus"].astype(str).unique()))

print("\nValores únicos de modalidad:")
print(sorted(df["modalidad"].astype(str).unique()))

In [ ]:
# 8.4 Fuera de rango
fuera_promedio = df[~df["promedio"].between(0, 100, inclusive="both") & df["promedio"].notna()]
fuera_asistencia = df[~df["asistencia_pct"].between(0, 100, inclusive="both") & df["asistencia_pct"].notna()]

print("Promedio fuera de rango:")
display(fuera_promedio[["id_estudiante", "nombre", "promedio"]])

print("Asistencia fuera de rango:")
display(fuera_asistencia[["id_estudiante", "nombre", "asistencia_pct"]])

In [ ]:
# 8.5 Error lógico: aprobados > inscritos
error_creditos = df[df["creditos_aprobados"] > df["creditos_inscritos"]]
display(error_creditos[["id_estudiante", "creditos_inscritos", "creditos_aprobados"]])

## 9) Limpieza básica paso a paso

Estrategia simple:
- corregir texto
- eliminar duplicados
- corregir rangos
- corregir lógica
- imputar faltantes con mediana

In [ ]:
df_limpio = df.copy()

# 9.1 Quitar espacios
df_limpio["campus"] = df_limpio["campus"].astype(str).str.strip()
df_limpio["modalidad"] = df_limpio["modalidad"].astype(str).str.strip()

# 9.2 Normalizar y corregir etiquetas
df_limpio["campus"] = df_limpio["campus"].str.lower().map({
    "slp centro": "SLP Centro",
    "slp cntro": "SLP Centro",
    "slp norte": "SLP Norte",
    "matehuala": "Matehuala"
}).fillna(df_limpio["campus"])

df_limpio["modalidad"] = df_limpio["modalidad"].str.lower().map({
    "escolarizada": "Escolarizada",
    "escolar": "Escolarizada",
    "mixta": "Mixta"
}).fillna(df_limpio["modalidad"])

# 9.3 Eliminar duplicados por id
df_limpio = df_limpio.drop_duplicates(subset=["id_estudiante"], keep="first").copy()

# 9.4 Corregir rangos
df_limpio["promedio"] = df_limpio["promedio"].clip(0, 100)
df_limpio["asistencia_pct"] = df_limpio["asistencia_pct"].clip(0, 100)

# 9.5 Corregir error lógico
df_limpio["creditos_aprobados"] = np.minimum(df_limpio["creditos_aprobados"], df_limpio["creditos_inscritos"])

# 9.6 Imputación simple con mediana
for col in ["edad", "promedio", "asistencia_pct"]:
    df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())

print("Limpieza completada ✅")
display(df_limpio)

In [ ]:
# Verificación después de limpiar
reporte = pd.DataFrame({
    "criterio": [
        "faltantes_totales",
        "duplicados_por_id",
        "campus_invalidos",
        "modalidad_invalidos",
        "promedio_fuera_rango",
        "asistencia_fuera_rango",
        "error_logico_creditos"
    ],
    "valor": [
        int(df_limpio.isna().sum().sum()),
        int(df_limpio.duplicated(subset=["id_estudiante"]).sum()),
        int((~df_limpio["campus"].isin(["SLP Centro", "SLP Norte", "Matehuala"])).sum()),
        int((~df_limpio["modalidad"].isin(["Escolarizada", "Mixta"])).sum()),
        int((~df_limpio["promedio"].between(0, 100, inclusive="both")).sum()),
        int((~df_limpio["asistencia_pct"].between(0, 100, inclusive="both")).sum()),
        int((df_limpio["creditos_aprobados"] > df_limpio["creditos_inscritos"]).sum()),
    ]
})
reporte

## 10) Filtración de datos
Filtrar = quedarnos con filas que cumplen condiciones.

In [ ]:
# Estudiantes con promedio menor a 80
df_limpio[df_limpio["promedio"] < 80][["id_estudiante", "nombre", "promedio"]]

In [ ]:
# SLP Centro y modalidad Escolarizada
df_limpio[(df_limpio["campus"] == "SLP Centro") & (df_limpio["modalidad"] == "Escolarizada")]

## 11) Transformación de datos
Crear nuevas variables para apoyar el análisis.

In [ ]:
# 11.1 Tasa de aprobación
df_limpio["tasa_aprobacion"] = (df_limpio["creditos_aprobados"] / df_limpio["creditos_inscritos"]).round(2)

# 11.2 Riesgo académico (regla simple)
df_limpio["riesgo_academico"] = np.select(
    [
        (df_limpio["promedio"] < 70) | (df_limpio["asistencia_pct"] < 75),
        (df_limpio["promedio"] < 80)
    ],
    ["Alto", "Medio"],
    default="Bajo"
)

display(df_limpio[["id_estudiante", "promedio", "asistencia_pct", "tasa_aprobacion", "riesgo_academico"]])

In [ ]:
# 11.3 One-hot encoding básico (modalidad)
dummies_modalidad = pd.get_dummies(df_limpio["modalidad"], prefix="modalidad", dtype=int)
display(dummies_modalidad.head())

df_transformado = pd.concat([df_limpio, dummies_modalidad], axis=1)
df_transformado.head()

## 12) Fusión de datos con `merge()`
Unimos estudiantes con la tabla de apoyos.

In [ ]:
df_final = df_limpio.merge(apoyos, on="id_estudiante", how="left", indicator=True)

print("Tamaño final:", df_final.shape)
display(df_final["_merge"].value_counts())
display(df_final)

**Interpretación de `_merge`:**
- `both`: aparece en ambas tablas
- `left_only`: sólo aparece en estudiantes

## 13) Cuantificación final (resumen)

In [ ]:
print("Promedio general:", round(df_final["promedio"].mean(), 2))
print("Asistencia promedio:", round(df_final["asistencia_pct"].mean(), 2))

print("\nDistribución de riesgo:")
display(df_final["riesgo_academico"].value_counts())

print("\nPromedio por campus:")
display(df_final.groupby("campus")["promedio"].mean().round(2))

## 14) Guardar el archivo limpio

In [ ]:
Path("salida").mkdir(exist_ok=True)
df_final.to_csv("salida/estudiantes_preprocesados.csv", index=False)

print("Archivo guardado ✅")
print("Ruta: salida/estudiantes_preprocesados.csv")

## 15) Mini reto (nivel licenciatura)

Con base en `df_final`, realiza:
1. Muestra estudiantes con `riesgo_academico == "Alto"`.
2. Cuenta cuántos tienen beca (`beca == "Sí"`).
3. Calcula el promedio de `promedio` por `modalidad`.
4. Crea una columna `aprobado_general`:
   - `"Sí"` si `promedio >= 70`
   - `"No"` en otro caso

In [ ]:
# Espacio para resolver el mini reto
# Escribe aquí tu solución

## 16) Cierre y reflexión

1. ¿Qué errores fueron más fáciles de corregir?
2. ¿Qué decisión de limpieza podría cambiar según el contexto?
3. ¿Por qué conviene guardar el archivo limpio por separado?

> Un buen análisis empieza con datos bien preparados.